In [5]:
# 📦 1. Importar librerías
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Instalar AutoGluon si es necesario
# !pip install autogluon.timeseries

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

# ⚙️ 2. Configuración del ensemble
# Estos parámetros pueden ajustarse según tus necesidades
MAGIC_PRODUCTS = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021, 20026, 20028, 
                 20035, 20039, 20042, 20044, 20045, 20046, 20049, 20051, 20052, 20053, 
                 20055, 20008, 20001, 20017, 20086, 20180, 20193, 20320, 20532, 20612, 
                 20637, 20807, 20838]

# Número de productos adicionales que usarán AutoGluon
NUM_AUTOGLUON_PRODUCTS = 700 # el mejor fue 500

# Seleccionar primeros o últimos productos para AutoGluon
# True = últimos productos, False = primeros productos
USE_LAST_PRODUCTS = False  # Cambia a False si prefieres usar los primeros 

# Número de meses para calcular el promedio histórico
HISTORICAL_MONTHS = 14

# Tiempo límite para entrenamiento de AutoGluon (en segundos)
AUTOGLUON_TIME_LIMIT = 60*60  # 10 minutos

# 📄 3. Cargar datasets
print("Cargando datos...")
df_sellin = pd.read_csv("../datasets/sell-in.txt", sep="\t")
df_productos = pd.read_csv("../datasets/tb_productos.txt", sep="\t")

# Leer lista de productos a predecir
with open("../datasets/product_id_apredecir201912.txt", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

print(f"Total de productos a predecir: {len(product_ids)}")

# 🧹 4. Preprocesamiento general
print("Preparando datos...")
# Convertir periodo a datetime para ambos modelos
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')
df_sellin['periodo_num'] = df_sellin['periodo']

# Filtrar hasta dic 2019 y productos requeridos
df_filtered = df_sellin[
    (df_sellin['timestamp'] <= '2019-12-01') & 
    (df_sellin['product_id'].isin(product_ids))
]

# Dataset agregado por periodo y producto
df_monthly_product = df_filtered.groupby(['timestamp', 'product_id'], as_index=False)['tn'].sum()

# 🧮 5. Preparación para Regresión Lineal
print("Preparando datos para Regresión Lineal...")
# Dataset para regresión lineal
df_reg = df_filtered.groupby(['periodo_num', 'product_id'])['tn'].sum().reset_index()

# Crear features (lags) y target (t+2)
df_reg['target'] = df_reg.groupby('product_id')['tn'].shift(-2)  # t+2 (Feb 2020)
for i in range(1, HISTORICAL_MONTHS+1):
    df_reg[f'tn_{i}'] = df_reg.groupby('product_id')['tn'].shift(i)

# Features para el modelo
features = ['tn'] + [f'tn_{i}' for i in range(1, HISTORICAL_MONTHS+1)]

# 🤖 6. Entrenamiento del modelo de Regresión Lineal
print("Entrenando modelo de Regresión Lineal...")
# Entrenar con datos de diciembre 2018
train_data = df_reg[(df_reg['periodo_num'] == 201812) & 
                    (df_reg['product_id'].isin(MAGIC_PRODUCTS))].dropna()

if len(train_data) > 0:
    X_train = train_data[features]
    y_train = train_data['target']
    
    linear_model = LinearRegression()
    linear_model.fit(X_train, y_train)
    
    print(f"Modelo de regresión entrenado con {len(train_data)} registros")
else:
    print("No hay suficientes datos para entrenar el modelo de regresión lineal.")
    linear_model = None

# 🤖 7. Preparación y entrenamiento de AutoGluon
print("Preparando datos para AutoGluon...")
# Determinar productos para AutoGluon (después de los mágicos)
remaining_products = [p for p in product_ids if p not in MAGIC_PRODUCTS]

if USE_LAST_PRODUCTS:
    autogluon_products = remaining_products[-NUM_AUTOGLUON_PRODUCTS:]
else:
    autogluon_products = remaining_products[:NUM_AUTOGLUON_PRODUCTS]

print(f"Productos para AutoGluon: {len(autogluon_products)}")

# Filtrar solo productos para AutoGluon
df_autogluon = df_monthly_product[df_monthly_product['product_id'].isin(autogluon_products)].copy()
df_autogluon['item_id'] = df_autogluon['product_id']

# Crear TimeSeriesDataFrame
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_autogluon,
    id_column='item_id',
    timestamp_column='timestamp'
)

# Completar valores faltantes
ts_data = ts_data.fill_missing_values()

print("Entrenando modelo AutoGluon...")
# Definir y entrenar predictor de AutoGluon
predictor = TimeSeriesPredictor(
    prediction_length=2,  # Predecir 2 meses adelante
    target='tn',
    freq='MS'  # Frecuencia mensual (Month Start)
)

if len(autogluon_products) > 0 and not ts_data.empty:
    predictor.fit(
        ts_data, 
        num_val_windows=2, 
        time_limit=AUTOGLUON_TIME_LIMIT
    )
    print("Modelo AutoGluon entrenado exitosamente")
    
    # Generar predicción con AutoGluon
    forecast = predictor.predict(ts_data)
    autogluon_preds = forecast['mean'].reset_index()
    # Filtrar solo febrero 2020
    autogluon_preds = autogluon_preds[autogluon_preds['timestamp'] == '2020-02-01']
    # Seleccionar columnas relevantes
    autogluon_preds = autogluon_preds[['item_id', 'mean']]
    autogluon_preds.columns = ['product_id', 'tn']
else:
    print("No hay suficientes datos para entrenar AutoGluon")
    autogluon_preds = pd.DataFrame(columns=['product_id', 'tn'])

# 📊 8. Preparar promedio histórico para productos restantes
print("Calculando promedios históricos...")
# Calcular promedio de los últimos 12 meses (excluyendo ceros)
hist_start_date = pd.to_datetime('2019-01-01') - pd.DateOffset(months=HISTORICAL_MONTHS-1)
hist_data = df_monthly_product[
    (df_monthly_product['timestamp'] >= hist_start_date) & 
    (df_monthly_product['timestamp'] <= '2019-12-01')
]

promedios = hist_data[hist_data['tn'] > 0].groupby('product_id')['tn'].mean().reset_index()

# 🔄 9. Crear ensemble final
print("Generando predicciones del ensemble...")
# Inicializar lista para predicciones
predicciones = []

# Datos de diciembre 2019 para hacer predicciones con regresión
dec_2019 = df_reg[df_reg['periodo_num'] == 201912].copy()

# Para cada producto a predecir
for product_id in product_ids:
    prediction = None
    
    # Estrategia 1: Usar Regresión Lineal para productos mágicos
    if product_id in MAGIC_PRODUCTS and linear_model is not None:
        product_data = dec_2019[dec_2019['product_id'] == product_id]
        
        if not product_data.empty and not product_data[features].isnull().any().any():
            X_pred = product_data[features]
            prediction = linear_model.predict(X_pred)[0]
            prediction = max(0, prediction)  # No permitir predicciones negativas
            predicciones.append({'product_id': product_id, 'tn': prediction, 'model': 'linear_regression'})
            continue
    
    # Estrategia 2: Usar AutoGluon para los siguientes productos
    if prediction is None and product_id in autogluon_products:
        ag_pred = autogluon_preds[autogluon_preds['product_id'] == product_id]
        if not ag_pred.empty:
            prediction = ag_pred['tn'].values[0]
            predicciones.append({'product_id': product_id, 'tn': prediction, 'model': 'autogluon'})
            continue
    
    # Estrategia 3: Usar promedio histórico para el resto
    if prediction is None:
        avg_pred = promedios[promedios['product_id'] == product_id]
        if not avg_pred.empty:
            prediction = avg_pred['tn'].values[0]
        else:
            prediction = 0  # Si no hay datos históricos
        
        predicciones.append({'product_id': product_id, 'tn': prediction, 'model': 'historical_avg'})

# Crear DataFrame final
submission = pd.DataFrame(predicciones)

# 📊 10. Estadísticas y guardado
# Contar uso de cada modelo
linear_count = len(submission[submission['model'] == 'linear_regression'])
autogluon_count = len(submission[submission['model'] == 'autogluon'])
hist_avg_count = len(submission[submission['model'] == 'historical_avg'])

print(f"\n📊 Estadísticas del ensemble:")
print(f"- Predicciones con Regresión Lineal: {linear_count}")
print(f"- Predicciones con AutoGluon: {autogluon_count}")
print(f"- Predicciones con Promedio Histórico: {hist_avg_count}")
print(f"- Valor promedio predicho: {submission['tn'].mean():.4f}")
print(f"- Valor máximo predicho: {submission['tn'].max():.4f}")

# Preparar submission final (solo product_id y tn)
final_submission = submission[['product_id', 'tn']].copy()

# Guardar submission
final_submission.to_csv('ensemble_submission1.csv', index=False)
print("\n✅ Archivo 'ensemble_submission.csv' guardado exitosamente")

# Mostrar primeras filas
final_submission.head()

Cargando datos...
Total de productos a predecir: 780
Preparando datos...


Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/home/agrocopilot/labo3-2025r/Nootebook/AutogluonModels/ag-20250811_032023'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #12-Ubuntu SMP Wed Jul 16 03:18:29 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       48.38 GB / 60.79 GB (79.6%)
Disk Space Avail:   48.99 GB / 247.17 GB (19.8%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 3600,
 'verbosity': 2}



Preparando datos para Regresión Lineal...
Entrenando modelo de Regresión Lineal...
Modelo de regresión entrenado con 33 registros
Preparando datos para AutoGluon...
Productos para AutoGluon: 700
Entrenando modelo AutoGluon...


train_data with frequency 'IRREG' has been resampled to frequency 'MS'.
Provided train_data has 20292 rows (NaN fraction=0.1%), 700 time series. Median time series length is 36 (min=4, max=36). 
	Removing 70 short time series from train_data. Only series with length >= 9 will be used for training.
	After filtering, train_data has 19865 rows (NaN fraction=0.1%), 630 time series. Median time series length is 36 (min=9, max=36). 

Provided data contains following columns:
	target: 'tn'
	past_covariates:
		categorical:        []
		continuous (float): ['product_id']

To learn how to fix incorrectly inferred types, please see documentation for TimeSeriesPredictor.fit

AutoGluon will gauge predictive performance using evaluation metric: 'WQL'
	This metric's sign has been flipped to adhere to being higher_is_better. The metric score can be multiplied by -1 to get the metric value.

Starting training. Start time is 2025-08-11 03:20:25
Models that will be trained: ['SeasonalNaive', 'RecursiveTab

Modelo AutoGluon entrenado exitosamente


Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


Calculando promedios históricos...
Generando predicciones del ensemble...

📊 Estadísticas del ensemble:
- Predicciones con Regresión Lineal: 33
- Predicciones con AutoGluon: 700
- Predicciones con Promedio Histórico: 47
- Valor promedio predicho: 36.3759
- Valor máximo predicho: 1282.3594

✅ Archivo 'ensemble_submission.csv' guardado exitosamente


,product_id,tn
0,20001,1119.799923
1,20002,1282.359430
2,20003,769.514631
3,20004,502.923110
4,20005,460.384539


In [6]:
# 📦 1. Importar librerías
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

# Instalar AutoGluon si es necesario
# !pip install autogluon.timeseries

from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

# ⚙️ 2. Configuración del ensemble y modelos
MAGIC_PRODUCTS = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021, 20026, 20028, 
                 20035, 20039, 20042, 20044, 20045, 20046, 20049, 20051, 20052, 20053, 
                 20055, 20008, 20001, 20017, 20086, 20180, 20193, 20320, 20532, 20612, 
                 20637, 20807, 20838]

NUM_AUTOGLUON_PRODUCTS = 700
USE_LAST_PRODUCTS = False

HISTORICAL_MONTHS = 14
AUTOGLUON_TIME_LIMIT = 60 * 60  # 1 hora


LGBM_PARAMS = {
    'n_estimators': 150,
    'learning_rate': 0.1,
    'max_depth': -1,
    'random_state': 42,
}

# 📄 3. Cargar datasets
print("Cargando datos...")
df_sellin = pd.read_csv("../datasets/sell-in.txt", sep="\t")
df_productos = pd.read_csv("../datasets/tb_productos.txt", sep="\t")
with open("../datasets/product_id_apredecir201912.txt", "r") as f:
    product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

# 🧹 4. Preprocesamiento
df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')
df_sellin['periodo_num'] = df_sellin['periodo'].astype(int)

df_filtered = df_sellin[
    (df_sellin['timestamp'] <= '2019-12-01') & 
    (df_sellin['product_id'].isin(product_ids))
]
df_monthly_product = df_filtered.groupby(
    ['timestamp','product_id'], as_index=False)['tn'].sum()

# 🧮 5. Construir tabla para regresión y LGBM
df_reg = df_filtered.groupby(
    ['periodo_num','product_id'])['tn'].sum().reset_index()
df_reg['target'] = df_reg.groupby('product_id')['tn'].shift(-2)  # t+2
for i in range(1, HISTORICAL_MONTHS+1):
    df_reg[f'tn_{i}'] = df_reg.groupby('product_id')['tn'].shift(i)
features = ['tn'] + [f'tn_{i}' for i in range(1, HISTORICAL_MONTHS+1)]

# 🔢 6. Entrenar Regresión Lineal con MAGIC_PRODUCTS
print("Entrenando Regresión Lineal...")
train_lr = df_reg[
    (df_reg['periodo_num'] == 201812) & 
    (df_reg['product_id'].isin(MAGIC_PRODUCTS))
].dropna(subset=features+['target'])
if not train_lr.empty:
    X_lr = train_lr[features]
    y_lr = train_lr['target']
    linear_model = LinearRegression().fit(X_lr, y_lr)
else:
    linear_model = None
    print("⚠️ No enough data for Linear Regression")

# 🤖 7. Entrenar LightGBM en todos los productos
print("Entrenando LightGBM...")
train_lgb = df_reg[
    (df_reg['periodo_num'] <= 201812) &
    df_reg['target'].notna()
].dropna(subset=features+['target'])
if not train_lgb.empty:
    X_lgb = train_lgb[features]
    y_lgb = train_lgb['target']
    lgb_model = LGBMRegressor(**LGBM_PARAMS).fit(X_lgb, y_lgb)
else:
    lgb_model = None
    print("⚠️ No enough data for LightGBM")

# 🤖 8. Preparar y entrenar AutoGluon
print("Preparando AutoGluon...")
remaining_products = [p for p in product_ids if p not in MAGIC_PRODUCTS]
if USE_LAST_PRODUCTS:
    autogluon_products = remaining_products[-NUM_AUTOGLUON_PRODUCTS:]
else:
    autogluon_products = remaining_products[:NUM_AUTOGLUON_PRODUCTS]

df_ag = df_monthly_product[
    df_monthly_product['product_id'].isin(autogluon_products)
].copy()
df_ag['item_id'] = df_ag['product_id']
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_ag, id_column='item_id', timestamp_column='timestamp'
).fill_missing_values()

predictor = TimeSeriesPredictor(
    prediction_length=2,
    target='tn',
    freq='MS'
)
if not ts_data.empty:
    predictor.fit(ts_data, num_val_windows=2, time_limit=AUTOGLUON_TIME_LIMIT)#,refit_full=True)
    forecast = predictor.predict(ts_data)
    ag_preds = forecast['mean'].reset_index()
    ag_preds = ag_preds[ag_preds['timestamp'] == '2020-02-01']
    ag_preds = ag_preds[['item_id','mean']].rename(
        columns={'item_id':'product_id','mean':'tn'}
    )
else:
    ag_preds = pd.DataFrame(columns=['product_id','tn'])

# 📊 9. Promedios históricos
hist_start = pd.to_datetime('2019-01-01') - pd.DateOffset(months=HISTORICAL_MONTHS-1)
hist_data = df_monthly_product[
    (df_monthly_product['timestamp'] >= hist_start) &
    (df_monthly_product['timestamp'] <= '2019-12-01') &
    (df_monthly_product['tn'] > 0)
]
promedios = hist_data.groupby('product_id')['tn'].mean().reset_index()

# 🔄 10. Ensamble final con 4 approaches
dec_2019 = df_reg[df_reg['periodo_num'] == 201912].copy()
predicciones = []

for pid in product_ids:
    pred = None
    # 1️⃣ AutoGluon
    if pid in ag_preds['product_id'].values:
        pred = float(ag_preds.loc[ag_preds['product_id']==pid, 'tn'])
        model_used = 'autogluon'
    # 2️⃣ Regresión Lineal
    elif pid in MAGIC_PRODUCTS and linear_model is not None:
        row = dec_2019[dec_2019['product_id']==pid]
        if not row.empty and not row[features].isnull().any().any():
            pred = linear_model.predict(row[features])[0]
            model_used = 'linear_regression'
    # 3️⃣ LightGBM
    if pred is None and lgb_model is not None:
        row = dec_2019[dec_2019['product_id']==pid]
        if not row.empty and not row[features].isnull().any().any():
            pred = lgb_model.predict(row[features])[0]
            model_used = 'lightgbm'
    # 4️⃣ Promedio histórico
    if pred is None:
        avg = promedios.loc[promedios['product_id']==pid, 'tn']
        pred = float(avg) if not avg.empty else 0.0
        model_used = 'historical_avg'
    predicciones.append({
        'product_id': pid,
        'tn': max(0, pred),
        'model': model_used
    })

submission = pd.DataFrame(predicciones)

# 📊 11. Estadísticas y guardado
counts = submission['model'].value_counts()
print("\n📊 Uso de cada modelo:")
for m, c in counts.items():
    print(f"- {m}: {c} productos")
print(f"- Valor promedio predicho: {submission['tn'].mean():.4f}")
print(f"- Valor máximo predicho: {submission['tn'].max():.4f}")

final_submission = submission[['product_id','tn']]
final_submission.to_csv('ensemble_submission2.csv', index=False)
print("\n✅ 'ensemble_submission.csv' guardado.")

Cargando datos...
Entrenando Regresión Lineal...
Entrenando LightGBM...
Preparando AutoGluon...


Beginning AutoGluon training... Time limit = 3600s
AutoGluon will save models to '/home/agrocopilot/labo3-2025r/Nootebook/AutogluonModels/ag-20250811_034649'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #12-Ubuntu SMP Wed Jul 16 03:18:29 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       51.29 GB / 60.79 GB (84.4%)
Disk Space Avail:   48.56 GB / 247.17 GB (19.6%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 3600,
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled t


📊 Uso de cada modelo:
- autogluon: 700 productos
- linear_regression: 33 productos
- lightgbm: 32 productos
- historical_avg: 15 productos
- Valor promedio predicho: 36.3899
- Valor máximo predicho: 1282.3594

✅ 'ensemble_submission.csv' guardado.


In [7]:
# ==============================================================================
# 📦 1. IMPORTAR LIBRERÍAS
# ==============================================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
import warnings
import json
import os
from datetime import datetime

warnings.filterwarnings('ignore')

try:
    from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
except ImportError:
    print("Instalando AutoGluon… esto puede tardar unos minutos.")
    # !pip install autogluon.timeseries
    from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

print("Librerías importadas correctamente.")

# ==============================================================================
# ⚙️ 2. PANEL DE CONTROL: CLASE DE CONFIGURACIÓN
# ==============================================================================
class Config:
    def __init__(self):
        self.paths = {
            "sellin": "../datasets/sell-in.txt",
            "pred_input": "../datasets/product_id_apredecir201912.txt",
            "output_dir": "submissions/"
        }
        self.experiment_name = f"Ensemble_Segmentado_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

        self.ensemble_strategy = {
            'linear_regression': 100,
            'lightgbm': 300,
            'autogluon': 300,
            'historical_avg': 'all_remaining'
        }
        
        self.linear_regression_magic_products = [
            20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021, 20026, 20028, 
            20035, 20039, 20042, 20044, 20045, 20046, 20049, 20051, 20052, 20053, 
            20055, 20008, 20001, 20017, 20086, 20180, 20193, 20320, 20532, 20612, 
            20637, 20807, 20838
        ]

        self.features = {
            "lags": [1, 2, 3, 6, 12],
            "rolling_windows": [3, 6, 12],
            "ewm_alphas": [0.3]
        }

        self.model_params = {
            "lightgbm": {
                'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 1000,
                'learning_rate': 0.05, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
                'bagging_freq': 1, 'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': 42
            },
            "autogluon_time_limit": 30 * 60,
            "autogluon_preset": 'medium_quality',
            "historical_avg_months": 12
        }

    def to_dict(self):
        conf_dict = self.__dict__.copy()
        for k, v in conf_dict.items():
            if isinstance(v, dict):
                for sub_k, sub_v in v.items():
                    if isinstance(sub_v, set):
                        v[sub_k] = list(sub_v)
        return conf_dict

cfg = Config()

# ==============================================================================
# 📝 3. FUNCIONES AUXILIARES
# ==============================================================================
def load_data(config):
    print("Cargando datasets…")
    try:
        df_sellin = pd.read_csv(config.paths["sellin"], sep="\t")
        with open(config.paths["pred_input"], "r") as f:
            product_ids = [int(line.strip()) for line in f if line.strip().isdigit()]
        return df_sellin, product_ids
    except FileNotFoundError as e:
        print(f"Error: No se encontró el archivo {e.filename}. Verifica las rutas en la Config.")
        return None, None

def create_features(df, config):
    print("Creando características…")
    df_features = df.sort_values(by=['product_id', 'timestamp']).copy()
    df_features['target'] = df_features.groupby('product_id')['tn'].shift(-2)

    for lag in config.features['lags']:
        df_features[f'lag_{lag}'] = df_features.groupby('product_id')['tn'].shift(lag)
    for w in config.features['rolling_windows']:
        df_features[f'rolling_mean_{w}'] = df_features.groupby('product_id')['tn'].shift(1).rolling(w).mean()
    for a in config.features['ewm_alphas']:
        df_features[f'ewm_alpha_{a}'] = df_features.groupby('product_id')['tn'].shift(1).ewm(alpha=a).mean()
        
    return df_features

def log_experiment(config, submission_df):
    output_dir = config.paths['output_dir']
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    log_path = os.path.join(output_dir, f"{config.experiment_name}_log.txt")

    with open(log_path, 'w') as f:
        f.write(f"EXPERIMENT LOG: {config.experiment_name}\n\n")
        f.write("Configuration:\n" + json.dumps(config.to_dict(), indent=4) + "\n\n")
        f.write("Submission Summary:\n" + f"  - Total Predictions: {len(submission_df)}\n")
        f.write(f"  - Mean Prediction: {submission_df['tn'].mean():.4f}\n")
        f.write("Model Counts:\n" + submission_df['model'].value_counts().to_string())
    print(f"Log del experimento guardado en: {log_path}")

# ==============================================================================
# 🚀 4. PIPELINE PRINCIPAL
# ==============================================================================
def main():
    df_sellin, product_ids_to_predict = load_data(cfg)
    if df_sellin is None:
        return

    df_sellin['timestamp'] = pd.to_datetime(df_sellin['periodo'], format='%Y%m')
    df_agg = df_sellin.groupby(['periodo', 'timestamp', 'product_id'])['tn'].sum().reset_index()
    df_model_data = create_features(df_agg, cfg)

    all_predictions = []
    remaining_products = list(product_ids_to_predict)
    features_list = [c for c in df_model_data.columns if c not in ['periodo', 'timestamp', 'product_id', 'target', 'tn']]

    for model_name, num_products in cfg.ensemble_strategy.items():
        if not remaining_products:
            break

        if num_products == 'all_remaining':
            current_products = remaining_products
        else:
            current_products = remaining_products[:num_products]
        
        print(f"\n--- Aplicando Modelo: {model_name.upper()} a {len(current_products)} productos ---")
        
        if model_name == 'linear_regression':
            train_df = df_model_data[
                (df_model_data['periodo'] == 201812) &
                (df_model_data['product_id'].isin(cfg.linear_regression_magic_products))
            ].dropna(subset=['target'])
            
            train_df[features_list] = train_df[features_list].fillna(0)
            predict_df = df_model_data[
                (df_model_data['periodo'] == 201912) &
                (df_model_data['product_id'].isin(current_products))
            ]
            predict_df[features_list] = predict_df[features_list].fillna(0)

            model = LinearRegression()
            model.fit(train_df[features_list], train_df['target'])
            predictions = model.predict(predict_df[features_list])
            preds_df = pd.DataFrame({'product_id': predict_df['product_id'], 'tn': predictions})

        elif model_name == 'lightgbm':
            train_df = df_model_data[df_model_data['periodo'] < 201912].dropna(subset=['target'])
            train_df[features_list] = train_df[features_list].fillna(0)
            predict_df = df_model_data[
                (df_model_data['periodo'] == 201912) &
                (df_model_data['product_id'].isin(current_products))
            ]
            predict_df[features_list] = predict_df[features_list].fillna(0)

            model = lgb.LGBMRegressor(**cfg.model_params['lightgbm'])
            model.fit(train_df[features_list], train_df['target'])
            predictions = model.predict(predict_df[features_list])
            preds_df = pd.DataFrame({'product_id': predict_df['product_id'], 'tn': predictions})
        
        elif model_name == 'autogluon':
            train_data = df_agg[df_agg['product_id'].isin(current_products)]
            ts_data = TimeSeriesDataFrame.from_data_frame(
                train_data, id_column='product_id', timestamp_column='timestamp'
            )
            predictor = TimeSeriesPredictor(
                prediction_length=2, path=f"{cfg.paths['output_dir']}/{cfg.experiment_name}_ag_models",
                target='tn', eval_metric='MASE', freq='MS'
            )
            predictor.fit(
                ts_data, 
                presets=cfg.model_params['autogluon_preset'], 
                time_limit=cfg.model_params['autogluon_time_limit']
            )
            forecast = predictor.predict(ts_data)
            forecast_reset = forecast.reset_index()

            # Identificar columna de fecha
            date_column = None
            for col in forecast_reset.columns:
                if any(word in col.lower() for word in ['timestamp', 'date', 'time', 'period']):
                    date_column = col
                    break
            if date_column is None:
                for col in forecast_reset.columns:
                    if pd.api.types.is_datetime64_any_dtype(forecast_reset[col]):
                        date_column = col
                        break

            target_date = pd.Timestamp('2020-02-01')
            preds_temp = forecast_reset[forecast_reset[date_column] == target_date]
            id_column = 'item_id' if 'item_id' in preds_temp.columns else 'product_id'
            pred_column = 'mean' if 'mean' in preds_temp.columns else preds_temp.select_dtypes(include=[np.number]).columns[0]
            preds_df = preds_temp[[id_column, pred_column]].rename(columns={id_column: 'product_id', pred_column: 'tn'})

        elif model_name == 'historical_avg':
            start_date = pd.to_datetime('201912', format='%Y%m') - pd.DateOffset(months=cfg.model_params['historical_avg_months']-1)
            hist_data = df_agg[
                (df_agg['timestamp'] >= start_date) & 
                (df_agg['product_id'].isin(current_products)) &
                (df_agg['tn'] > 0)
            ]
            preds_df = hist_data.groupby('product_id')['tn'].mean().reset_index()

        preds_df['model'] = model_name
        all_predictions.append(preds_df)
        remaining_products = [p for p in remaining_products if p not in current_products]
        
    print("\nEnsamblando predicciones finales...")
    final_df = pd.concat(all_predictions, ignore_index=True)

    submission_template = pd.DataFrame({'product_id': product_ids_to_predict})
    final_submission = pd.merge(submission_template, final_df, on='product_id', how='left')

    final_submission['tn'].fillna(0, inplace=True)
    final_submission['tn'] = final_submission['tn'].apply(lambda x: max(0, x))
    final_submission['model'].fillna('No_Prediction', inplace=True)

    submission_path = os.path.join(cfg.paths['output_dir'], f"{cfg.experiment_name}_submission.csv")
    final_submission[['product_id', 'tn']].to_csv(submission_path, index=False)
    print(f"\n✅ Archivo de sumisión guardado en: {submission_path}")

    log_experiment(cfg, final_submission)

# ==============================================================================
# 🏁 EJECUCIÓN
# ==============================================================================
if __name__ == '__main__':
    main()

Librerías importadas correctamente.
Cargando datasets…
Creando características…

--- Aplicando Modelo: LINEAR_REGRESSION a 100 productos ---

--- Aplicando Modelo: LIGHTGBM a 300 productos ---


Beginning AutoGluon training... Time limit = 1800s
AutoGluon will save models to '/home/agrocopilot/labo3-2025r/Nootebook/submissions/Ensemble_Segmentado_20250811_041108_ag_models'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #12-Ubuntu SMP Wed Jul 16 03:18:29 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       48.56 GB / 60.79 GB (79.9%)
Disk Space Avail:   48.18 GB / 247.17 GB (19.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'freq': 'MS',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'time_limit': 1800,
 'verbosity': 2


--- Aplicando Modelo: AUTOGLUON a 300 productos ---


Provided train_data has 7923 rows (NaN fraction=0.2%), 300 time series. Median time series length is 36 (min=4, max=36). 
	Removing 27 short time series from train_data. Only series with length >= 7 will be used for training.
	After filtering, train_data has 7777 rows (NaN fraction=0.2%), 273 time series. Median time series length is 36 (min=7, max=36). 

Provided data contains following columns:
	target: 'tn'
	past_covariates:
		categorical:        []
		continuous (float): ['periodo']

To learn how to fix incorrectly inferred types, please see documentation for TimeSeriesPredictor.fit

AutoGluon will gauge predictive performance using evaluation metric: 'MASE'
	This metric's sign has been flipped to adhere to being higher_is_better. The metric score can be multiplied by -1 to get the metric value.

Starting training. Start time is 2025-08-11 04:11:46
Models that will be trained: ['Naive', 'SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'ETS', 'Theta', 'Chronos[bolt_small]', 'Tem


--- Aplicando Modelo: HISTORICAL_AVG a 80 productos ---

Ensamblando predicciones finales...

✅ Archivo de sumisión guardado en: submissions/Ensemble_Segmentado_20250811_041108_submission.csv
Log del experimento guardado en: submissions/Ensemble_Segmentado_20250811_041108_log.txt
